## Setup

*You must run the cells in this section each time you connect to a new runtime. For example, when you return to the notebook after an idle timeout, when the runtime crashes, or when you restart or factory reset the runtime.*

Install requirements (*Note: ocdskingfishercolab installs google-colab, which expects specific versions of pandas and numpy*):

In [ ]:
! pip install --upgrade pip > pip.log
! pip install --upgrade ocdskingfishercolab ipywidgets psycopg2-binary >> pip.log

In [ ]:
# @title Import packages and load extensions { display-mode: "form" }

import gzip
import json
import os
import shutil
import tempfile
from collections import Counter
from datetime import datetime, timezone
from pathlib import Path

import numpy as np
import pandas as pd
from dateutil.relativedelta import relativedelta
from google.colab.data_table import DataTable
from google.colab.files import download
from ipywidgets import widgets
from ocdskingfishercolab import (
    authenticate_gspread,
    calculate_coverage,
    download_dataframe_as_csv,
    format_thousands,
    indicator_checks,
    load_indicators,
    most_common_fields_to_calculate_indicators,
    render_json,
    save_dataframe_to_sheet,
    save_dataframe_to_spreadsheet,
    set_dark_mode,
    set_light_mode,
)

# Load https://pypi.org/project/ipython-sql/
%load_ext sql
# Load https://colab.research.google.com/notebooks/data_table.ipynb
%load_ext google.colab.data_table

In [ ]:
# @title Configure the notebook environment { display-mode: "form" }

# Increase max columns so that Pandas DataFrames with many columns are rendered as data tables.
DataTable.max_columns = 50
# Remove the index from data tables for easier copy-pasting to Google Docs.
DataTable.include_index = False

# Return Pandas DataFrames instead of regular result sets.
%config SqlMagic.autopandas = True
# Don't print number of rows affected.
%config SqlMagic.feedback = False

# If you set Tools > Settings > Site > Theme to dark, uncomment this line.
# set_dark_mode()
# If you are creating plots to copy-paste into reports, uncomment this line.
# set_light_mode()

## Setup Kingfisher Process

### Connect to the database

In [ ]:
import getpass

from ocdskingfishercolab import (
    list_collections,
    list_source_ids,
    set_search_path,
)


Enter your PostgreSQL credentials and connect to the Kingfisher Process database:

In [ ]:
user = input("Username:")
password = getpass.getpass("Password:")
connection_string = (
    "postgresql://"
    + user
    + ":"
    + password
    + "@postgres.kingfisher.open-contracting.org/kingfisher_process?sslmode=require"
)
# Don't show connection string after execute.
%config SqlMagic.displaycon = False
%sql $connection_string

### Choose collections and schema

*Use this section to choose the collections and schema that you want to query.*

#### Set the collection(s)

Update `collection_ids` with the `id`(s) of the [Kingfisher Process collection(s)](https://kingfisher-process.readthedocs.io/en/latest/data-model.html#collections):

In [ ]:
collection_ids = (2358, 2359)

If you don't know which collections you need, run the next cell and use the **Filter** button to filter the [collection table](https://kingfisher-process.readthedocs.io/en/latest/database-structure.html#collection-table) to find the collection(s). You can use the `source_id` column to filter on the `name` of the [Kingfisher Collect spider](https://kingfisher-collect.readthedocs.io/en/latest/spiders.html) used to collect the data. Use the value(s) from the `id` column to update the previous cell.

In [ ]:
list_collections()

#### Set the schema

Update `schema_name` with the name of the [Kingfisher Summarize schema](https://kingfisher-summarize.readthedocs.io/en/latest/index.html#how-it-works).

In [ ]:
schema_name = "view_data_collection_2358_2359"
set_search_path(schema_name)

If you don't know which schema you need, run the next cell and use the **Filter** button to filter the [selected collections table](https://kingfisher-summarize.readthedocs.io/en/latest/database.html#summaries-selected-collections) to find the schema. You can use the `collection_id` column to filter on the `id` of the collections that you identified in the previous step. Alternatively, you can filter on the `source_id` column. Use the value from the `schema` column to update the previous cell.

In [ ]:
%%sql
SELECT
    summaries.selected_collections.*,
    source_id
FROM
    summaries.selected_collections
INNER JOIN
    collection
    ON summaries.selected_collections.collection_id = collection.id


If you can't find a schema containing the collections that you want to query, you can create a schema using [Kingfisher Summarize](https://ocdsdeploy.readthedocs.io/en/latest/use/kingfisher-summarize.html).

## Calculate general statistics

Use this section to have an overview of the dataset in terms of fields published, stages covered and key summary statistics.  You can add any additional queries as needed.

### Fields published

Calculate the fields published

In [ ]:
%%sql fields_table <<
SELECT
    path,
    distinct_releases
FROM
    field_counts
WHERE
    release_type = 'compiled_release'


In [ ]:
fields_table

Save fields table in spreadsheet:

In [ ]:
spreadsheet_name = input("Enter the name of your spreadsheet:")
save_dataframe_to_sheet(spreadsheet_name, fields_table, "fields")

### Stages covered

In [ ]:
%%sql stages <<
WITH field_counts AS (
    SELECT *
    FROM
        field_counts
    WHERE
        release_type = 'compiled_release'
        AND path IN (
            'planning',
            'tender',
            'awards',
            'contracts',
            'contracts/implementation'
        )
)

SELECT
    CASE
        WHEN paths.path = 'contracts/implementation'
            THEN
                'implementation'
        ELSE
            paths.path
    END AS stage,
    CASE
        WHEN
            paths.path IN ('planning', 'tender', 'contracts/implementation')
            THEN
                greatest(field_counts.object_property, 0)
        ELSE
            greatest(field_counts.array_count, 0)
    END AS object_count
FROM (
    SELECT
        unnest(
            ARRAY[
                'planning',
                'tender',
                'awards',
                'contracts',
                'contracts/implementation'
            ]
        ) AS path
) AS paths
LEFT JOIN field_counts USING (path)


In [ ]:
stages

In [ ]:
plot_objects_per_stage(stages)

### Number of tenders and awards by date

In [ ]:
%%sql dates <<
WITH tenders AS (
    SELECT
        extract(YEAR FROM tenderperiod_startdate) AS year,
        count(*) AS tenders
    FROM
        tender_summary
    WHERE
        release_type = 'compiled_release'
        AND tenderperiod_startdate IS NOT NULL
    GROUP BY
        year
    ORDER BY
        year
),

awards AS (
    SELECT
        extract(YEAR FROM date) AS year,
        count(*) AS awards
    FROM
        awards_summary
    WHERE
        release_type = 'compiled_release'
        AND date IS NOT NULL
    GROUP BY
        year
    ORDER BY
        year
)

SELECT
    t.year::integer,
    tenders,
    awards
FROM
    tenders AS t
FULL JOIN awards AS a ON t.year = a.year


In [ ]:
dates

In [ ]:
plot_objects_per_year(dates)

### Procurement methods used

In [ ]:
%%sql
SELECT
    procurementmethod,
    count(DISTINCT ocid),
    round((count(*) * 100.0 / sum(count(*)) OVER ()), 1) AS proportion
FROM
    tender_summary
WHERE
    release_type = 'compiled_release'
GROUP BY
    procurementmethod
ORDER BY
    proportion DESC


In [ ]:
%%sql
SELECT
    tender ->> 'procurementMethodDetails' AS method,
    count(DISTINCT ocid),
    round((count(*) * 100.0 / sum(count(*)) OVER ()), 1) AS proportion
FROM
    tender_summary
WHERE
    release_type = 'compiled_release'
GROUP BY
    method
ORDER BY
    proportion DESC


### Number of procedures by buyer

In [ ]:
%%sql buyers <<
SELECT
    identifier AS party_id,
    party -> 'name' AS name,
    count(DISTINCT ocid) AS total_tenders
FROM
    parties_summary
WHERE
    roles::text ILIKE '%%buyer%%'
    AND release_type = 'compiled_release'
GROUP BY
    identifier,
    party -> 'name'
ORDER BY
    total_tenders DESC


In [ ]:
buyers.head(10)

In [ ]:
plot_top_buyers(buyers.head(10))

## Usability analysis

Generate a list of the fields published:

In [ ]:
fields_list = set(fields_table["path"])
indicators = load_indicators(prefix="U")
result = indicator_checks(fields_list, indicators)
result = result.rename(columns={"id": "U_id"})
result["coverage"] = get_coverage(fields_list, indicators)

### Export and visualize results

#### Choose language of the export

In [ ]:
lang = widgets.Dropdown(options=["Spanish", "English"], description="language", style={"description_width": "initial"})
lang

#### Load use case indicators spreadsheet

In [ ]:
result_final = check_usability_indicators(lang, result)

#### Table of results

In [ ]:
result_final

#### Most common fields for indicators

This table shows the most frequent fields used to calculate indicators and if they are published.  You can use this table to highlight to the publisher the key data gaps.

In [ ]:
fields_count = most_common_fields_to_calculate_indicators(fields_list, indicators)
fields_count

#### Save tables to spreadsheet

In [ ]:
spreadsheet_name = input("Enter the name of your spreadsheet:")
save_dataframe_to_sheet(spreadsheet_name, result_final, "usability table")
save_dataframe_to_sheet(spreadsheet_name, fields_count, "key fields")

#### Visualize results

In [ ]:
plot_usability_indicators(result_final, lang.value)